# CFRP Anomaly Detection — Modellvergleich

Vier unüberwachte Detektoren auf dem CFRP-Datensatz:

- **Isolation Forest** — baumbasiert; günstig, starker Default.
- **LOF** (Local Outlier Factor) — dichtebasiert.
- **ECOD** — empirische CDF, parameterfrei.
- **AutoEncoder** (PyTorch) — neuronal; 49→32→16→32→49.

Jedes Modell vergibt einen kontinuierlichen Anomalie-Score. Die Entscheidungs-Schwelle wird auf dem **Validierungs-Split** durch Maximierung von F1 gewählt und anschließend auf dem unberührten **Test-Split** ausgewertet. Schwellwert-unabhängige Metriken (ROC-AUC, PR-AUC) werden parallel berichtet.

Jeder Lauf wird in MLflow (`./mlruns`) protokolliert.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import mlflow

from src.data_loader import load_cfrp, split_70_20_10
from src.preprocessing import prepare
from src.modeling import build_detectors, run_baseline
from src.mlflow_utils import load_config, init_mlflow, log_split_sizes

config = load_config(PROJECT_ROOT / "config.yaml")
init_mlflow(config)


## 1. Daten laden und aufteilen

In [ ]:
raw_path = PROJECT_ROOT / config["dataset"]["raw_path"]
X, y = load_cfrp(raw_path)
print(f"X={X.shape}  y={y.shape}  anomaly rate={y.mean():.4%}")

X_train, y_train, X_val, y_val, X_test, y_test = split_70_20_10(
    X, y,
    seed=config["project"]["random_seed"],
    stratify=config["split"]["stratify_on_label"],
)

split_table = pd.DataFrame(
    {
        "n": [len(y_train), len(y_val), len(y_test)],
        "share": [len(y_train) / len(y), len(y_val) / len(y), len(y_test) / len(y)],
        "pos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
        "pos_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "val", "test"],
)
split_table

## 2. Datenvorverarbeitung (Zuerst imputieren, dann skalieren,  Nur auf den Trainingsdaten anpassen)

In [ ]:
data = prepare(X_train, y_train, X_val, y_val, X_test, y_test,
               impute="median", scaler="robust")
print("Shapes after preprocessing:")
print(f"  train: {data.X_train.shape}")
print(f"  val:   {data.X_val.shape}")
print(f"  test:  {data.X_test.shape}")

## 3. Größen der Daten‑Splits zentral protokolliert – in einem übergeordneten parenterun

In [ ]:
with mlflow.start_run(run_name="data_setup"):
    mlflow.set_tag("stage", "data")
    log_split_sizes(data.X_train, data.X_val, data.X_test,
                    data.y_train, data.y_val, data.y_test)
    mlflow.log_param("impute_strategy", "median")
    mlflow.log_param("scaler", "robust")
    mlflow.log_param("random_seed", config["project"]["random_seed"])
    mlflow.log_artifact(str(PROJECT_ROOT / "reports" / "dataset_card.md"),
                        artifact_path="dataset")

## 4. Training und Evaluation aller vier Modelle

Jeder Aufruf unten ist ein eigener MLflow-Run (Parameter, Metriken, Verwechslungsmatrix, ROC-Kurve, gefittetes Modell, Dataset-Card).


In [ ]:
contamination = float(data.y_train.mean())
detectors = build_detectors(contamination=contamination,
                            seed=config["project"]["random_seed"])

results = {}
for name, det in detectors.items():
    print(f"\n=== {name} ===")
    extra_params = {}
    if name == "autoencoder":
        extra_params = {
            "hidden_neuron_list": str(det.hidden_neuron_list),
            "epoch_num": det.epoch_num,
            "batch_size": det.batch_size,
            "dropout_rate": det.dropout_rate,
            "lr": det.lr,
        }
    res = run_baseline(name, det, data, params={"contamination": contamination, **extra_params})
    results[name] = res
    print(f"  val  ROC-AUC={res.val.roc_auc:.4f}  PR-AUC={res.val.pr_auc:.4f}  F1={res.val.f1:.4f}")
    print(f"  test ROC-AUC={res.test.roc_auc:.4f}  PR-AUC={res.test.pr_auc:.4f}  F1={res.test.f1:.4f}")

## 5. Vergleichstabelle

In [ ]:
rows = []
for name, res in results.items():
    rows.append({
        "model": name,
        "val_ROC-AUC": res.val.roc_auc,
        "val_PR-AUC": res.val.pr_auc,
        "val_F1": res.val.f1,
        "test_ROC-AUC": res.test.roc_auc,
        "test_PR-AUC": res.test.pr_auc,
        "test_precision": res.test.precision,
        "test_recall": res.test.recall,
        "test_F1": res.test.f1,
        "threshold": res.test.threshold,
        "tp": res.test.tp, "fp": res.test.fp,
        "fn": res.test.fn, "tn": res.test.tn,
    })
summary = pd.DataFrame(rows).set_index("model")
summary.style.format("{:.4f}", subset=[c for c in summary.columns if c not in ("tp", "fp", "fn", "tn")])

## 7. Hyperparameter Sweep — Isolation Forest

Es wurde ein Hyperparameter‑Sweep mit insgesamt zwölf Kombinationen der Parameter n_estimators, max_samples und max_features durchgeführt. Jede Parameterkombination wird als eigenständiger MLflow‑Run protokolliert und mit dem Tag sweep=iforest_v1 versehen, sodass alle Sweep‑Runs in der MLflow‑Benutzeroberfläche gemeinsam gefiltert und analysiert werden können.

- TRAIN/VAL/TEST split, sDatenvorverarbeitung (Skalierung) sowie der Zufallsseed bleiben für alle Runs unverändert
- Innerhalb jedes Runs wird die F1‑optimale Entscheidungsschwelle ausschließlich auf dem Validierungssplit neu bestimmt, sodass kein Informationsleck aus dem Testdatensatz entsteht.
- Die Auswahl des besten Modells erfolgt auf Basis des Validierungs‑F1‑Wertes; dessen Leistungskennzahlen werden anschließend auf dem Testdatensatz berichtet. Eine Auswahl anhand des Test‑F1‑Wertes würde einer methodisch unzulässigen Nutzung der Testdaten (Data Leakage) entsprechen und wird daher bewusst vermieden.

In [ ]:
from itertools import product
from pyod.models.iforest import IForest

grid = {
    "n_estimators": [100, 200, 400],
    "max_samples":  [256, 1024],
    "max_features": [0.5, 1.0],
}

seed = config["project"]["random_seed"]
sweep_rows = []

for n_est, n_samp, n_feat in product(grid["n_estimators"], grid["max_samples"], grid["max_features"]):
    run_name = f"iforest_n{n_est}_s{n_samp}_f{n_feat}"
    print(f"=== {run_name} ===")
    det = IForest(
        n_estimators=n_est,
        max_samples=n_samp,
        max_features=n_feat,
        contamination=contamination,
        random_state=seed,
        n_jobs=-1,
    )
    res = run_baseline(
        run_name,
        det,
        data,
        params={
            "contamination": contamination,
            "n_estimators": n_est,
            "max_samples": n_samp,
            "max_features": n_feat,
        },
        tags={"sweep": "iforest_v1"},
        model_family="iforest",
    )
    sweep_rows.append({
        "run": run_name,
        "n_estimators": n_est,
        "max_samples": n_samp,
        "max_features": n_feat,
        "val_F1": res.val.f1,
        "test_F1": res.test.f1,
        "test_ROC-AUC": res.test.roc_auc,
        "test_PR-AUC": res.test.pr_auc,
        "test_precision": res.test.precision,
        "test_recall": res.test.recall,
    })
    print(f"  val F1={res.val.f1:.4f}  test F1={res.test.f1:.4f}  test PR-AUC={res.test.pr_auc:.4f}")

sweep_df = pd.DataFrame(sweep_rows).sort_values("val_F1", ascending=False).reset_index(drop=True)
print("\nTop 3 by validation F1:")
print(sweep_df.head(3).to_string(index=False))
sweep_df.style.format("{:.4f}", subset=[c for c in sweep_df.columns if c not in ("run", "n_estimators", "max_samples")])

## 6. Ergebnisse ansehen in MLflow

```
mlflow ui --backend-store-uri file:./mlruns
```

Then open http://127.0.0.1:5000 — every run carries:

- parameters (model name, contamination, scaler, seed),
- val + test metrics (ROC-AUC, PR-AUC, threshold, precision, recall, F1, confusion-matrix counts),
- artifacts: confusion-matrix image, ROC image, fitted model (joblib), dataset card.